# PathBench ArtP and DArtP: reproducible Colab run

This notebook is the source of truth for a **linear Runtime → Run all** validation on one real CC0 recording. Select a GPU runtime first. It uses only public HTTPS resources and disables Git terminal prompts. It validates representative real data; **it does not reproduce the complete PathBench benchmark table**.

ArtP is reference-based: it aligns the supplied transcript's phonemes to phonetic-model emissions. DArtP is reference-free at scoring time: English ASR plus a pinned n-gram decoder first supplies a transcript, then the same phonetic alignment stage scores it. Lower-quality alignment generally produces a lower average aligned-phoneme probability; these fixture values are regression expectations, not calibrated clinical interpretations.

In [ ]:
# Public, noninteractive checkout. Override PATHBENCH_REVISION to test a PR SHA.
import os, pathlib, shutil, subprocess
os.environ["GIT_TERMINAL_PROMPT"] = "0"
REPOSITORY = "https://github.com/karkirowle/pathbench.git"
REVISION = os.environ.get("PATHBENCH_REVISION", "main")
ROOT = pathlib.Path("/content/pathbench")
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(["git", "clone", "--filter=blob:none", REPOSITORY, str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", REVISION], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Exact repository revision:", COMMIT)


## Runtime identities
The model identifiers below are the evaluator defaults. The later cells also print the installed model paths/configurations and the authenticated language-model digest.

In [ ]:
import platform, shutil, subprocess
print("Python:", platform.python_version(), platform.python_implementation())
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("PyTorch CUDA runtime:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
except ImportError: print("PyTorch: not installed yet")
print("nvidia-smi:")
subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], check=True)
print("espeak-ng before pinned install:", subprocess.run(["espeak-ng", "--version"], text=True, capture_output=True).stdout.strip() if shutil.which("espeak-ng") else "not installed")
print("Phonetic model (ArtP and DArtP): facebook/wav2vec2-xlsr-53-espeak-cv-ft")
print("DArtP ASR model: jonatasgrosman/wav2vec2-large-xlsr-53-english")
print("DArtP language model: Zenodo record 18738598, lms/wiki_en_token.arpa.bin")


## Native and GPU prerequisites
This calls the repository's existing installer/validator rather than recreating setup logic. It pins espeak-ng commit `2ea41210`, creates `tools/gpu_venv`, installs CUDA PyTorch, confirms GPU access, and runs the focused tests. At this early stage DArtP is expected to skip because its large language model is deliberately downloaded later; the final regression cell rejects every skip.

In [ ]:
subprocess.run(["python3", "tools/test_gpu_predictors.py", "--install-system-dependencies"], check=True)
VENV_PY = ROOT / "tools/gpu_venv/bin/python"
subprocess.run([str(VENV_PY), "-c", "import torch,sys; print('Python:',sys.version); print('PyTorch:',torch.__version__); print('CUDA:',torch.version.cuda); print('GPU:',torch.cuda.get_device_name(0))"], check=True)
print("Pinned espeak-ng marker:", pathlib.Path("/usr/local/share/pathbench/espeak-ng-commit").read_text().strip())
subprocess.run(["espeak-ng", "--version"], check=True)


## Recording integrity, license, and provenance
These four committed single-word **BLUE** recordings came from the public `Bartelds/neural-acoustic-distance` corpus and are distributed under the **CC0-1.0 public-domain dedication**. We authenticate the bytes before inference. The selected input is `BLUE_japanese10.wav`; the intended transcript is `blue`.

In [ ]:
from pathlib import Path
from tools.colab_bootstrap import (BLUE_LICENSE, BLUE_PROVENANCE_URL, TRANSCRIPT, verify_blue_recordings)
print("License:", BLUE_LICENSE)
print("Provenance:", BLUE_PROVENANCE_URL)
print("Input transcript:", repr(TRANSCRIPT))
for name, digest in verify_blue_recordings(Path("tests/data")).items(): print(name, digest)
AUDIO = str(Path("tests/data/BLUE_japanese10.wav").resolve())


## ArtP: direct evaluator API
ArtP consumes the known transcript `blue` with language `en-us`. Its score is the mean probability on the forced-aligned reference phonemes. The test expectation is `0.0695`, with absolute tolerance `0.00005` (the interval represented by the project's four-decimal `assertAlmostEqual`). Small scores are not percentages or diagnoses.

In [ ]:
artp_program = r'''
from pathbench.articulatory_precision_evaluator import ArticulatoryPrecisionEvaluator
from tools.colab_bootstrap import *
AUDIO = "tests/data/BLUE_japanese10.wav"
config = {"evaluator":"ArticulatoryPrecisionEvaluator", "phonetic_model":"facebook/wav2vec2-xlsr-53-espeak-cv-ft", "utterance_id":"BLUE_japanese10", "audio":AUDIO, "transcription":TRANSCRIPT, "language":"en-us", "start_time":0.0, "end_time":-1.0}
print("Exact ArtP configuration:", config)
print("Input transcript:", repr(TRANSCRIPT))
score = ArticulatoryPrecisionEvaluator(model_id=config["phonetic_model"]).score(config["utterance_id"], AUDIO, TRANSCRIPT, config["language"], start_time=0.0, end_time=-1.0)
print("ArtP numeric score:", score, "expected:", ARTP_EXPECTED, "absolute tolerance:", SCORE_TOLERANCE)
assert_score("ArtP", score, ARTP_EXPECTED, SCORE_TOLERANCE)
'''
subprocess.run([str(VENV_PY), "-c", artp_program], check=True, cwd=ROOT)


## ⚠️ EXPENSIVE: acquire the pinned English n-gram model (~13.6 GiB expanded)
The cell prints required and available space **before** network transfer. The helper range-extracts only the 13.6 GiB member from the immutable Zenodo record, validates ZIP metadata/CRC/SHA-256, and atomically installs it. Expected SHA-256: `d786eec55174c696c0bf3327928ff496684f482194ba3c6ebdf4311acb823d00`.

In [ ]:
from tools.test_gpu_predictors import LANGUAGE_MODEL_SIZE
from tools.colab_bootstrap import disk_space
required, available = disk_space(ROOT, LANGUAGE_MODEL_SIZE)
print(f"EXPENSIVE DOWNLOAD PREFLIGHT — required (model + 1 GiB margin): {required:,} bytes ({required/1024**3:.2f} GiB)")
print(f"Available: {available:,} bytes ({available/1024**3:.2f} GiB)")
print("Starting authenticated ~8.0 GiB transfer / ~13.6 GiB expansion now.")
install_program = r'''
from pathlib import Path
from tools.test_gpu_predictors import *
venv_python = Path("tools/gpu_venv/bin/python").resolve()
model = install_zenodo_language_model(validator=lambda p: validate_language_model(venv_python, p))
assert sha256(model) == LANGUAGE_MODEL_SHA256
print("Language-model path:", model.resolve())
print("Language-model identity (SHA-256):", sha256(model))
'''
subprocess.run([str(VENV_PY), "-c", install_program], check=True, cwd=ROOT)


## DArtP: direct evaluator API
DArtP runs the pinned English wav2vec2 ASR logits through the authenticated KenLM n-gram decoder, phonemizes that **ASR transcript**, and force-aligns it against the phonetic model. We wrap `decoder.decode` only to display its exact returned transcript; scoring still calls the evaluator API directly. Expected score: `0.4405 ± 0.00005`.

In [ ]:
dartp_program = r'''
from pathbench.artp_double_asr_evaluator import ArtPDoubleASREvaluator
from tools.colab_bootstrap import *
AUDIO = "tests/data/BLUE_japanese10.wav"
config = {"evaluator":"ArtPDoubleASREvaluator", "phonetic_model":"facebook/wav2vec2-xlsr-53-espeak-cv-ft", "asr_model":"jonatasgrosman/wav2vec2-large-xlsr-53-english", "language_model":"lms/wiki_en_token.arpa.bin", "language_model_sha256":"d786eec55174c696c0bf3327928ff496684f482194ba3c6ebdf4311acb823d00", "utterance_id":"BLUE_japanese10", "audio":AUDIO, "language":"en-us", "start_time":0.0, "end_time":-1.0}
print("Exact DArtP configuration:", config)
print("Input/intended transcript (not passed to reference-free DArtP):", repr(TRANSCRIPT))
evaluator = ArtPDoubleASREvaluator(language=config["language"], model_id=config["phonetic_model"] )
captured = {}
original_decode = evaluator.decoder.decode
def displaying_decode(logits):
    transcript = original_decode(logits); captured["transcript"] = transcript
    print("ASR + n-gram transcript:", repr(transcript)); return transcript
evaluator.decoder.decode = displaying_decode
score = evaluator.score(config["utterance_id"], AUDIO, start_time=0.0, end_time=-1.0)
assert "transcript" in captured
print("DArtP numeric score:", score, "expected:", DARTP_EXPECTED, "absolute tolerance:", SCORE_TOLERANCE)
assert_score("DArtP", score, DARTP_EXPECTED, SCORE_TOLERANCE)
'''
subprocess.run([str(VENV_PY), "-c", dartp_program], check=True, cwd=ROOT)


## Focused real-data regressions (skips are failures)
The two project tests call the same evaluator APIs on the authenticated recording. JUnit is inspected separately so a pytest success containing a skip cannot be mistaken for validation.

In [ ]:
report = ROOT / "focused-artp-dartp.xml"
focused = [str(VENV_PY), "-m", "pytest", "tests/test_evaluators.py::TestEvaluatorMethods::test_articulatory_precision", "tests/test_evaluators.py::TestEvaluatorMethods::test_artp_double_asr", "-v", f"--junitxml={report}"]
subprocess.run(focused, check=True, cwd=ROOT, env={**os.environ, "MPLBACKEND":"Agg"})
from tools.colab_bootstrap import assert_junit_no_skips
assert_junit_no_skips(report)
print("PASS: both focused real-data regressions ran; zero skips, failures, or errors.")


## Scope, cleanup, and reruns
This validates representative real data and the ArtP/DArtP paths; it **does not reproduce the complete benchmark table**, which requires all datasets and benchmark orchestration. Colab storage and GPU assignments are ephemeral. To reclaim space now, run the cleanup cell. For a clean rerun, use **Runtime → Disconnect and delete runtime**, select a GPU again, and **Runtime → Run all**. Model downloads and Hugging Face caches will not survive a deleted runtime.

In [ ]:
# Optional cleanup: uncomment, then run. A subsequent Run all reclones and reacquires everything.
# import shutil
# shutil.rmtree("/content/pathbench", ignore_errors=True)
# shutil.rmtree("/root/.cache/huggingface", ignore_errors=True)
print("Cleanup is intentionally disabled. Uncomment the lines above to remove checkout, 13.6 GiB LM, venv, and model cache.")
